# M4 — MiniLM + SVM Linéaire

Ce modèle utilise MiniLM (all-MiniLM-L6-v2) pour représenter les énoncés sous forme d'embeddings, puis un SVM linéaire pour prédire les tags de chaque exercice. Les énoncés longs sont découpés en blocs dont les représentations sont agrégées, sans modifier les poids de MiniLM. Après l'entraînement, plusieurs configurations d'hyperparamètres sont comparées sur l'ensemble de validation afin de sélectionner la meilleure selon le F1 macro. Les seuils de décision sont ensuite ajustés pour chaque tag, puis le modèle est évalué et sauvegardé. L'ensemble de test reste réservé à l'évaluation finale.

## 1. Charger et contrôler les données

Les descriptions, entrées, sorties et notes forment les seules variables explicatives.

In [1]:
import hashlib
import html
import json
import os
import platform
import unicodedata
import warnings
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import joblib
import torch
# Aucune barre de progression interactive n’est utilisée.
warnings.filterwarnings("ignore", message="IProgress not found.*")
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    precision_recall_fscore_support,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

racine = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "entrainement.jsonl").is_file()), None
)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

In [2]:
dossier_modele = racine / "models" / "m4_minilm_linear_svm"
dossier_modele.mkdir(parents=True, exist_ok=True)
graine = 42
np.random.seed(graine)
torch.manual_seed(graine)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
champs_texte = {"prob_desc_description": "DESCRIPTION", "prob_desc_input_spec": "ENTREES",
                "prob_desc_output_spec": "SORTIES", "prob_desc_notes": "NOTES"}
versions = {nom: version(nom) for nom in
            ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "threadpoolctl",
             "torch", "transformers", "sentence-transformers", "tokenizers", "huggingface-hub", "safetensors"]}
empreintes = {}

def charger_ensemble(nom):
    chemin = racine / "data" / f"{nom}.jsonl"
    contenu = chemin.read_bytes()
    empreintes[nom] = hashlib.sha256(contenu).hexdigest()
    return pd.DataFrame([json.loads(ligne) for ligne in contenu.decode("utf-8").splitlines()
                         if ligne.strip()])

entrainement = charger_ensemble("entrainement")
validation = charger_ensemble("validation")
for donnees in [entrainement, validation]:
    assert donnees[colonnes_cibles].isin([0, 1]).all().all(), "Cibles non binaires."
    assert set(champs_texte).issubset(donnees.columns)
y_train = entrainement[colonnes_cibles].to_numpy(dtype=np.int64)
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)
for champ in ["fichier_source", "src_uid", "code_uid"]:
    assert not entrainement[champ].isna().any() and not validation[champ].isna().any()
    assert not entrainement[champ].duplicated().any()
    assert not validation[champ].duplicated().any()
    assert set(entrainement[champ]).isdisjoint(validation[champ]), f"Fuite : {champ}"
for cibles in [y_train, y_validation]:
    assert ((cibles.sum(axis=0) > 0) & (cibles.sum(axis=0) < len(cibles))).all()

warnings.filterwarnings("error", category=ConvergenceWarning)
print(f"{len(entrainement)} exercices d’entraînement et {len(validation)} de validation.")

3470 exercices d’entraînement et 739 de validation.


## 2. Construire les embeddings MiniLM

Pour chaque exercice, la description, les spécifications d’entrée et de sortie et les notes sont réunies en un seul texte. MiniLM est figé pour que aucune information du corpus ne modifie ses poids. Le texte est tokenisé sans troncature puis découpé en blocs de 254 tokens de contenu,
sans chevauchement. Les blocs sont encodés en petits lots et leurs vecteurs sont moyennés en les pondérant par leur nombre de tokens de contenu, puis normalisés en L2. Chaque exercice conserve
ainsi tout son texte et possède un vecteur de 384 valeurs.

In [3]:
def construire_enonce(exercice):
    sections = []
    for champ, titre in champs_texte.items():
        valeur = exercice.get(champ)
        if isinstance(valeur, str) and valeur.strip():
            contenu = unicodedata.normalize("NFC", html.unescape(valeur)).strip()
            sections.append(f"{titre}\n{contenu}")
    return "\n\n".join(sections)

textes_train = entrainement.apply(construire_enonce, axis=1)
textes_validation = validation.apply(construire_enonce, axis=1)
assert textes_train.str.strip().ne("").all() and textes_validation.str.strip().ne("").all()

In [5]:
nom_encodeur = "sentence-transformers/all-MiniLM-L6-v2"
revision_encodeur = "fa97f6e7cb1a59073dff9e6b13e2715cf7475ac9"
parametres_encodage = {"taille_bloc": 254, "taille_lot": 16}
encodeur = SentenceTransformer(nom_encodeur, revision=revision_encodeur, device="cpu",
                              model_kwargs={"attn_implementation": "eager"})
encodeur.requires_grad_(False)
encodeur.eval()
assert encodeur.max_seq_length == 256
assert encodeur.get_sentence_embedding_dimension() == 384
assert encodeur.tokenizer.num_special_tokens_to_add(pair=False) == 2

def decouper_textes(textes, encodeur, taille_bloc=254):
    # Conserver les identifiants de tokens : aucun décodage/réencodage des blocs.
    contenus = encodeur.tokenizer(list(textes), add_special_tokens=False,
                                 truncation=False, padding=False, verbose=False)["input_ids"]
    if any(not contenu for contenu in contenus):
        raise ValueError("Un énoncé ne contient aucun token de contenu.")
    if not 1 <= taille_bloc <= encodeur.max_seq_length - 2:
        raise ValueError("La taille des blocs dépasse la capacité de MiniLM.")
    blocs, proprietaires, poids = [], [], []
    for indice, contenu in enumerate(contenus):
        for debut in range(0, len(contenu), taille_bloc):
            morceau = contenu[debut:debut + taille_bloc]
            ids = [
                encodeur.tokenizer.cls_token_id,
                *morceau,
                encodeur.tokenizer.sep_token_id
            ]
        
            blocs.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids)
            })
            proprietaires.append(indice)
            poids.append(len(morceau))
    longueurs = np.array([len(contenu) for contenu in contenus])
    proprietaires, poids = np.array(proprietaires), np.array(poids)
    # Le nombre de tokens conservés doit correspondre au texte initial.
    np.testing.assert_array_equal(
        np.bincount(proprietaires, weights=poids, minlength=len(contenus)), longueurs
    )
    assert all(len(bloc["input_ids"]) <= 256 for bloc in blocs)
    return blocs, proprietaires, poids, longueurs

def encoder_textes(textes, encodeur, taille_bloc=254, taille_lot=16):
    blocs, proprietaires, poids, longueurs = decouper_textes(textes, encodeur, taille_bloc)
    vecteurs = np.zeros((len(longueurs), 384), dtype=np.float64)
    encodeur.eval()
    # Les blocs sont déjà tokenisés : le padding séparé est volontaire.
    encodeur.tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = True
    with torch.inference_mode():
        for debut in range(0, len(blocs), taille_lot):
            fin = debut + taille_lot
            lot = encodeur.tokenizer.pad(blocs[debut:fin], padding=True, return_tensors="pt")
            lot = {cle: valeur.to(encodeur.device) for cle, valeur in lot.items()}
            embeddings = encodeur(lot)["sentence_embedding"].cpu().numpy()
            np.add.at(vecteurs, proprietaires[debut:fin], embeddings * poids[debut:fin, None])
    vecteurs /= longueurs[:, None]
    normes = np.linalg.norm(vecteurs, axis=1, keepdims=True)
    assert np.isfinite(vecteurs).all() and (normes > 0).all()
    vecteurs = (vecteurs / normes).astype(np.float32)
    return vecteurs, longueurs, np.bincount(proprietaires, minlength=len(longueurs))

debut = perf_counter()
X_train, longueurs_train, blocs_train = encoder_textes(textes_train, encodeur, **parametres_encodage)
X_validation, longueurs_validation, blocs_validation = encoder_textes(
    textes_validation, encodeur, **parametres_encodage
)
duree_encodage = perf_counter() - debut
assert X_train.shape == (len(entrainement), 384)
assert X_validation.shape == (len(validation), 384)
for matrice in [X_train, X_validation]:
    np.testing.assert_allclose(np.linalg.norm(matrice, axis=1), 1, atol=1e-6)
assert not any(parametre.requires_grad for parametre in encodeur.parameters())
display(pd.DataFrame([
    {"ensemble": nom, "exercices": len(longueurs),
     "depassent_256_tokens": int((longueurs + 2 > 256).sum()),
     "blocs_encodes": int(blocs.sum())}
    for nom, longueurs, blocs in [("Entraînement", longueurs_train, blocs_train),
                                 ("Validation", longueurs_validation, blocs_validation)]
]))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16401.42it/s]
/var/folders/8s/fjskjb050nsf6qwhyq994z500000gn/T/ipykernel_3160/1949590429.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert encodeur.get_sentence_embedding_dimension() == 384


,ensemble,exercices,depassent_256_tokens,blocs_encodes
0,Entraînement,3470,3247,9816
1,Validation,739,681,2028


## 3. Comparer six configurations

Huit classifieurs binaires sont entraînés via `OneVsRestClassifier`.
Les six couples `C ∈ {0.1, 1, 10}` et `class_weight ∈ {None, balanced}`
sont comparés avec le seuil par défaut. Le F1 macro donne le même poids
aux huit tags, le F1 micro complète la lecture globale.

In [6]:
def evaluer_predictions(reels, predictions):
    return {
        "f1_macro": float(f1_score(reels, predictions, average="macro", zero_division=0)),
        "f1_micro": float(f1_score(reels, predictions, average="micro", zero_division=0)),
        "hamming_loss": float(hamming_loss(reels, predictions)),
        "exact_match_ratio": float(accuracy_score(reels, predictions)),
    }

In [7]:
seuil_defaut = 0.0

def creer_classifieur(C, ponderation):
    estimateur = LinearSVC(C=C, class_weight=ponderation, dual="auto", max_iter=10000, random_state=graine)
    return OneVsRestClassifier(estimateur, n_jobs=1)

def calculer_scores(classifieur, matrice):
    return classifieur.decision_function(matrice)

experiences = []
classifieurs = {}
with threadpool_limits(limits=1):
    for C in [0.1, 1.0, 10.0]:
        for nom_ponderation, ponderation in [("aucune", None), ("equilibree", "balanced")]:
            cle = f"C{C}_{nom_ponderation}"
            classifieur = creer_classifieur(C, ponderation)
            debut = perf_counter()
            classifieur.fit(X_train, y_train)
            duree = perf_counter() - debut
            scores = calculer_scores(classifieur, X_validation)
            predictions = (scores >= seuil_defaut).astype(int)
            classifieurs[cle] = classifieur
            experiences.append({"experience": cle, "C": C, "class_weight": nom_ponderation,
                                 "temps_entrainement_s": duree,
                                 **evaluer_predictions(y_validation, predictions)})

recherche = pd.DataFrame(experiences).sort_values(
    ["f1_macro", "C"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
configuration_retenue = recherche.iloc[0]
classifieur_final = classifieurs[configuration_retenue["experience"]]
scores_validation = calculer_scores(classifieur_final, X_validation)
display(recherche.round(4))
print("Configuration retenue :", configuration_retenue["experience"])

,experience,C,class_weight,temps_entrainement_s,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,C10.0_aucune,10.0,aucune,1.0246,0.5483,0.4913,0.0739,0.5277
1,C1.0_aucune,1.0,aucune,0.7614,0.5442,0.4924,0.0680,0.5440
2,C1.0_equilibree,1.0,equilibree,1.0880,0.5237,0.5163,0.1106,0.3789
3,C0.1_equilibree,0.1,equilibree,0.7593,0.5089,0.5305,0.1120,0.3762
4,C10.0_equilibree,10.0,equilibree,2.1072,0.5016,0.4947,0.1123,0.3654
5,C0.1_aucune,0.1,aucune,0.6513,0.2974,0.3977,0.0722,0.5223


Configuration retenue : C10.0_aucune


## 4. Ajuster les huit seuils

Le classifieur retenu est conservé sans réentraînement. Pour chaque tag, choisir
le seuil donnant le meilleur F1 parmi une grille de quantiles des scores de validation
et le seuil par défaut (0.0).

In [8]:
quantiles_seuils = [0.50, 0.70, 0.80, 0.85, 0.90, 0.925,
                   0.95, 0.965, 0.975, 0.985, 0.99, 0.995]
seuils = []
for indice in range(len(tags_cibles)):
    candidats = np.unique(np.append(
        np.quantile(scores_validation[:, indice], quantiles_seuils), seuil_defaut
    ))
    evaluations = [
        (f1_score(y_validation[:, indice], scores_validation[:, indice] >= seuil,
                  zero_division=0), float(seuil)) for seuil in candidats
    ]
    meilleur = min(evaluations, key=lambda r: (-r[0], abs(r[1] - seuil_defaut), r[1]))
    seuils.append(meilleur[1])
seuils = np.array(seuils)

## 5. Lire les résultats de validation

Comparer les seuils par défaut et ajustés, puis examiner chaque tag avec son support,
sa précision, son rappel et son F1. 

In [9]:
lignes_globales, lignes_tags = [], []
for nom, valeur in [("Aucun tag", 0), ("Tous les tags", 1)]:
    predictions = np.full_like(y_validation, valeur)
    lignes_globales.append({"regle": nom, **evaluer_predictions(
        y_validation, predictions
    )})
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = (scores_validation >= seuils_regle).astype(int)
    lignes_globales.append({"regle": regle,
                           **evaluer_predictions(y_validation, predictions)})
    precisions, rappels, f1, supports = precision_recall_fscore_support(
        y_validation, predictions, average=None, zero_division=0
    )
    for i, tag in enumerate(tags_cibles):
        lignes_tags.append({
            "regle": regle, "tag": tag, "support": int(supports[i]),
            "seuil": float(seuils_regle[i]), "precision": precisions[i],
            "rappel": rappels[i], "f1": f1[i],
        })
bilan = pd.DataFrame(lignes_globales)
par_tag = pd.DataFrame(lignes_tags)
display(bilan.round(4))
display(par_tag.loc[par_tag["regle"].eq("Ajustés")].round(4).reset_index(drop=True))

sans_cible = y_validation.sum(axis=1) == 0
controle_sans_cible = []
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = scores_validation[sans_cible] >= seuils_regle
    avec_prediction = predictions.any(axis=1)
    controle_sans_cible.append({
        "regle": regle,
        "exercices_sans_cible": int(sans_cible.sum()),
        "aucun_tag_predit": int((~avec_prediction).sum()),
        "au_moins_un_tag_predit": int(avec_prediction.sum()),
    })
print("Contrôle des exercices sans tag cible (décomptes) :")
display(pd.DataFrame(controle_sans_cible))

,regle,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,Aucun tag,0.0000,0.0000,0.0864,0.4587
1,Tous les tags,0.1500,0.1591,0.9136,0.0000
2,Défaut,0.5483,0.4913,0.0739,0.5277
3,Ajustés,0.5787,0.5466,0.0937,0.4032


,regle,tag,support,seuil,precision,rappel,f1
0,Ajustés,math,211,-0.5047,0.4270,0.7488,0.5439
1,Ajustés,graphs,81,-0.2594,0.5946,0.5432,0.5677
2,Ajustés,strings,63,-0.3623,0.5676,0.6667,0.6131
3,Ajustés,number theory,52,-0.4658,0.4054,0.5769,0.4762
4,Ajustés,trees,49,-0.0379,0.6486,0.4898,0.5581
5,Ajustés,geometry,25,-0.7810,0.2679,0.6000,0.3704
6,Ajustés,games,16,0.0000,0.7500,0.7500,0.7500
7,Ajustés,probabilities,14,0.0000,0.9000,0.6429,0.7500


Contrôle des exercices sans tag cible (décomptes) :


,regle,exercices_sans_cible,aucun_tag_predit,au_moins_un_tag_predit
0,Défaut,339,267,72
1,Ajustés,339,132,207


## 6. Sauvegarder le modèle final et les résultats

Un unique artefact contient l’encodeur MiniLM avec son tokenizer, les huit classifieurs, les seuils,
l’ordre des tags et les règles de construction du texte. Les trois CSV conservent
la recherche, les métriques globales et le détail par tag. Le modèle sauvegardé
reste celui appris sur l’entraînement, les seuils ont été réglés sur la validation.

In [10]:
modele_final = {
    "modele": "M4", "encodeur": encodeur, "classifieur": classifieur_final,
    "seuils": seuils, "tags": tags_cibles, "champs_texte": champs_texte,
    "type_scores": "marges_de_decision",
    "pretraitement": {"html_unescape": True, "normalisation_unicode": "NFC",
                      "separateur_sections": "\n\n", "separateur_titre": "\n"},
    "configuration": {"C": float(configuration_retenue["C"]),
                      "class_weight": None if configuration_retenue["class_weight"] == "aucune"
                      else "balanced"},
    "nom_encodeur": nom_encodeur, "revision_encodeur": revision_encodeur,
    "encodage": {**parametres_encodage, "agregation": "moyenne_ponderee_par_tokens",
                 "normalisation": "l2", "dimension": 384},
    "materiel": {"appareil": "cpu", "architecture": platform.machine(), "threads_torch": 4},
    "graine": graine, "versions": versions, "python": platform.python_version(),
    "empreintes_donnees": empreintes,
}
_ = joblib.dump(modele_final, dossier_modele / "modele.joblib", compress=3)
recherche.to_csv(dossier_modele / "recherche_hyperparametres.csv", index=False)
bilan.to_csv(dossier_modele / "metriques_validation.csv", index=False)
par_tag.to_csv(dossier_modele / "metriques_par_tag_validation.csv", index=False)